# Stroke Prediction Neural Network

## set up Packages

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

## set training variables

In [3]:
args = {}
args['batch_size'] = 64
args['test_batch_size'] = 64
args['epochs'] = 100
args['lr'] = 0.05
args['momentum'] = 0.5
args['seed'] = 1
args['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args['log_interval'] = 10

## Load dataset

In [4]:
import pandas as pd

# Import Data
df = pd.read_csv('data/healthcare-dataset-stroke-data.csv')
df = df.drop(columns=['id'])
print(df.head())

   gender   age  hypertension  heart_disease ever_married      work_type  \
0    Male  67.0             0              1          Yes        Private   
1  Female  61.0             0              0          Yes  Self-employed   
2    Male  80.0             0              1          Yes        Private   
3  Female  49.0             0              0          Yes        Private   
4  Female  79.0             1              0          Yes  Self-employed   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             202.21   NaN     never smoked       1  
2          Rural             105.92  32.5     never smoked       1  
3          Urban             171.23  34.4           smokes       1  
4          Rural             174.12  24.0     never smoked       1  


## Data Preprocessing

In [5]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#convert categorical columns into numerical for the model
X = pd.get_dummies(df.drop(columns=['stroke']), drop_first=True).astype(np.float32) #features
y = df['stroke'].astype(np.int64) #labels

#evenly distribute datapoints
X_trainval, X_test, Y_trainval, Y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=args['seed'])

X_train, X_val, y_train, y_val = train_test_split(X_trainval, Y_trainval, test_size=0.25, stratify=Y_trainval, random_state=args['seed'])

X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
bmi_median = X_train['bmi'].median()
for i in (X_train, X_val, X_test):#fill the missing BMI data with the average BMI
    i['bmi']= i['bmi'].fillna(bmi_median)

number_columns = ['age', 'avg_glucose_level', 'bmi']
scalar = StandardScaler().fit(X_train[number_columns])
for i in (X_train, X_val, X_test):
    i[number_columns] = scalar.transform(i[number_columns])

#convert to tensor
X_train_tensor = torch.from_numpy(X_train.to_numpy(dtype=np.float32))
Y_train_tensor = torch.from_numpy(np.asarray(y_train, dtype=np.int64))

X_val_tensor = torch.from_numpy(X_val.to_numpy(dtype=np.float32))
Y_val_tensor = torch.from_numpy(y_val.to_numpy(dtype=np.int64))

X_test_tensor = torch.from_numpy(X_test.to_numpy(dtype=np.float32))
Y_test_tensor = torch.from_numpy(np.asarray(Y_test, dtype=np.int64))

#datasets
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=args['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=args['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=args['test_batch_size'], shuffle=False)

print(f"Training Samples: {len(train_dataset)}, Test Samples: {len(test_dataset)}")

Training Samples: 3066, Test Samples: 1022


C:\Users\benka\AppData\Local\Temp\ipykernel_20272\2528265871.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  Y_train_tensor = torch.from_numpy(np.asarray(y_train, dtype=np.int64))


## Create NN

In [6]:

class StrokePredictionNet(nn.Module):
    def __init__(self, dropout=0.4):
        super(StrokePredictionNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)



## Train the Model

In [7]:

def train(epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if args['device'] == 'cuda':
            data, target = data.cuda(), target.cuda()

        optimizer.zero_grad()
        output = model(data)

        loss = F.binary_cross_entropy_with_logits(output, target.float(), pos_weight=pos_weight)
        loss.backward()
        optimizer.step()

        if batch_idx % args['log_interval'] == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

## Evaluate the model

In [8]:
from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score)

#calculate probability on dataset
def calculate_probability(loader):
    model.eval()
    probs, labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(args['device']), target.to(args['device'])
            output = model(data)
            total_loss += F.binary_cross_entropy_with_logits(
                output, target.float(), reduction='sum').item()
            probs.append(torch.sigmoid(output).cpu())
            labels.append(target.cpu())
    probs = torch.cat(probs).numpy()
    labels = torch.cat(labels).numpy()
    return probs, labels, total_loss / len(loader.dataset)

def test(loader, threshold=0.5):
    probs, labels, avg_loss = calculate_probability(loader)
    preds = (probs >= threshold).astype(int)

    roc_auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    print(f"Stroke Test | loss={avg_loss:.4f} ROC={roc_auc:.3f} PR={pr_auc:.3f} "
              f"P={precision:.3f} "
              f"R={recall:.3f} "
              f"F1={f1:.3f}")
    print(confusion_matrix(labels, preds, labels=[0, 1]))


## Train and test model

In [9]:
from sklearn.metrics import confusion_matrix, roc_auc_score

model = StrokePredictionNet( dropout=0.1)#dropout helps prevent NN from memorizing training data
torch.manual_seed(args['seed'])
if args['device'] == 'cuda':
    model.cuda()
    torch.cuda.manual_seed(args['seed'])

optimizer = optim.AdamW(model.parameters(), lr=args['lr'], weight_decay=1e-2)

pos_weight = torch.tensor((y_train == 0).sum() / (y_train == 1).sum(),
                          dtype=torch.float32).to(args['device'])

for epoch in range(1, args['epochs'] + 1):
    train(epoch)
    test(val_loader, threshold=0.53)

exptorch.save(model.state_dict(), 'predictionNN.pth')

Train Epoch: 1 [0/3066 (0%)]	Loss: 1.302885
Train Epoch: 1 [640/3066 (21%)]	Loss: 1.336965
Train Epoch: 1 [1280/3066 (42%)]	Loss: 0.864504
Train Epoch: 1 [1920/3066 (62%)]	Loss: 1.206508
Train Epoch: 1 [2560/3066 (83%)]	Loss: 1.222800
Stroke Test | loss=0.6671 ROC=0.801 PR=0.133 P=0.098 R=0.960 F1=0.177
[[528 444]
 [  2  48]]
Train Epoch: 2 [0/3066 (0%)]	Loss: 0.737368
Train Epoch: 2 [640/3066 (21%)]	Loss: 0.657872
Train Epoch: 2 [1280/3066 (42%)]	Loss: 0.710717
Train Epoch: 2 [1920/3066 (62%)]	Loss: 0.947964
Train Epoch: 2 [2560/3066 (83%)]	Loss: 0.928572
Stroke Test | loss=0.5867 ROC=0.804 PR=0.131 P=0.104 R=0.940 F1=0.187
[[565 407]
 [  3  47]]
Train Epoch: 3 [0/3066 (0%)]	Loss: 0.929877
Train Epoch: 3 [640/3066 (21%)]	Loss: 1.124789
Train Epoch: 3 [1280/3066 (42%)]	Loss: 0.841382
Train Epoch: 3 [1920/3066 (62%)]	Loss: 1.184751
Train Epoch: 3 [2560/3066 (83%)]	Loss: 0.721361
Stroke Test | loss=0.4844 ROC=0.826 PR=0.165 P=0.127 R=0.860 F1=0.222
[[677 295]
 [  7  43]]
Train Epoch: 4 [

### check AUC on training data and validation

In [10]:
tr_p, tr_y, _ = calculate_probability(train_loader)
va_p, va_y, _ = calculate_probability(val_loader)
print(f"train AUC {roc_auc_score(tr_y, tr_p):.3f} | val AUC {roc_auc_score(va_y, va_p):.3f}")

train AUC 0.822 | val AUC 0.782
